# Train railway defect detection with YOLOv8 in Google Colab

This notebook trains a lightweight YOLOv8n detector from the supplied Roboflow YOLOv8 dataset, evaluates on the held-out test split, exports NCNN for Raspberry Pi 4, and packages the artifacts for download.

**Colab setup:** choose **Runtime → Change runtime type → T4 GPU** (or another available GPU). Training runs in Colab; inference will run locally on the Pi.

Classes are `defective` and `non-defective`. The dataset export says images were resized to 640×640, so the first baseline uses `imgsz=640`.


## 1. Enable GPU and install dependencies

In [ ]:
!nvidia-smi
!pip -q install ultralytics==8.3.40

## 2. Get the dataset ZIP
Choose upload or Drive. Comment out the method you do not use.

In [ ]:
# OPTION A: Upload the ZIP from your computer
from google.colab import files
uploaded = files.upload()  # select the dataset ZIP
ZIP_PATH = next(iter(uploaded))

# OPTION B: Read the ZIP from Google Drive instead
# from google.colab import drive
# drive.mount('/content/drive')
# ZIP_PATH = '/content/drive/MyDrive/RSDD Split 4.v2-split-4-with-augment.yolov8.zip'


In [ ]:
from pathlib import Path
import zipfile
zip_path = Path(ZIP_PATH)
assert zip_path.exists(), f'ZIP not found: {zip_path}'
extract_root = Path('/content/railway_dataset')
extract_root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(extract_root)
yamls = list(extract_root.rglob('data.yaml'))
assert len(yamls) == 1, f'Expected exactly one data.yaml; found {yamls}'
DATASET = yamls[0].parent
print('Dataset:', DATASET)
for split in ('train', 'valid', 'test'):
    print(split, len(list((DATASET / split / 'images').glob('*'))), 'images;', len(list((DATASET / split / 'labels').glob('*.txt'))), 'labels')


## 3. Create a Colab-correct data YAML

In [ ]:
import yaml
DATA_YAML = Path('/content/railway_data.yaml')
config = {
    'path': str(DATASET),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 2,
    'names': ['defective', 'non-defective'],
}
DATA_YAML.write_text(yaml.safe_dump(config, sort_keys=False))
print(DATA_YAML.read_text())


## 4. Train YOLOv8n
This is a small baseline suited to edge deployment. If Colab runs out of GPU memory, reduce `batch` to 8 or 4.

In [ ]:
import torch
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU is attached to this Colab runtime. In Colab choose "
        "Runtime > Change runtime type > T4 GPU, save, then reconnect and rerun. "
        "For a short CPU smoke test only, set DEVICE = 'cpu' and reduce epochs/batch below."
    )
DEVICE = 0

model = YOLO('yolov8n.pt')
model.train(
    data=str(DATA_YAML), epochs=100, imgsz=640, batch=16,
    patience=20, device=DEVICE, workers=4,
    project='/content/runs/railway', name='yolov8n_rail_defects',
    exist_ok=True, seed=42,
)
BEST_PT = Path('/content/runs/railway/yolov8n_rail_defects/weights/best.pt')
assert BEST_PT.exists(), BEST_PT
print('Best checkpoint:', BEST_PT)


## 5. Evaluate on the held-out test split

In [ ]:
best = YOLO(str(BEST_PT))
metrics = best.val(data=str(DATA_YAML), split='test', imgsz=640, batch=16, device=0)
print(metrics.results_dict)


## 6. Export NCNN for Raspberry Pi 4
Export in Colab, then transfer the folder to the Pi.

In [ ]:
exported = best.export(format='ncnn', imgsz=640)
print('NCNN export:', exported)


## 7. Package files for transfer to the Pi

In [ ]:
import shutil
from google.colab import files
PACKAGE = Path('/content/railway_pi_deployment')
if PACKAGE.exists(): shutil.rmtree(PACKAGE)
PACKAGE.mkdir(parents=True)
shutil.copy2(BEST_PT, PACKAGE / 'best.pt')
NCNN_DIR = Path(str(exported))
if not NCNN_DIR.is_dir():
    candidates = list(Path('/content/runs/railway/yolov8n_rail_defects/weights').glob('*ncnn*'))
    assert candidates, f'Could not find NCNN output: {exported}'
    NCNN_DIR = candidates[0]
shutil.copytree(NCNN_DIR, PACKAGE / NCNN_DIR.name)
results_dir = Path('/content/runs/railway/yolov8n_rail_defects')
for name in ('results.csv', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png'):
    p = results_dir / name
    if p.exists(): shutil.copy2(p, PACKAGE / name)
(PACKAGE / 'classes.txt').write_text('defective\nnon-defective\n')
archive = shutil.make_archive('/content/railway_pi_deployment', 'zip', PACKAGE)
print('Ready:', archive)
files.download(archive)


## Raspberry Pi 4 notes

- Use Raspberry Pi OS 64-bit. Verify the Camera Module 1.3 with `rpicam-hello`; install Picamera2 using `apt`.
- Unzip the deployment package on the Pi. Load the exported `*_ncnn_model` directory for inference, not `best.pt`.
- Begin with still images or sparse frame inference at a reduced camera resolution. Pi 4 is CPU-only; measure actual speed and detection quality before trying continuous video.
- Keep the test stationary first, with fixed camera position and lighting. Save annotated images locally.

**Review the `non-defective` class before training:** confirm its boxes are intentional and that you want normal track detected as an object. Some defect tasks label only defective regions and leave normal images as background. Use your annotation intent consistently.
